# Flux handwriting pilot v2: short free-Colab run (about 300 steps)

Goal: get a small real result. Train a LoRA on the 40 pilot word images for 300 steps, then generate test words and read them back with OCR.

Before you start
1. Runtime, Change runtime type, pick the T4 GPU (or better if you have it).
2. Put `words_pilot.zip` in the Colab Files panel (upload button) or at the top level of Google Drive (My Drive).
3. Run the cells in order. Keep this tab open. Checkpoints are saved to your Google Drive (`flux_out`), so they survive if the session ends.

If a cell fails, send me the error text.

In [ ]:
# 1) GPU check and Google Drive
import subprocess, shutil
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
else:
    print('NO GPU FOUND. Use Runtime > Change runtime type and pick a GPU, then run this cell again.')
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Load the 40 pilot images
import os, glob, shutil, subprocess
zp = '/content/words_pilot.zip' if os.path.exists('/content/words_pilot.zip') else '/content/drive/MyDrive/words_pilot.zip'
print('using', zp)
subprocess.run('rm -rf /content/words_pilot /content/words_pilot_raw', shell=True)
os.makedirs('/content/words_pilot', exist_ok=True)
subprocess.run(['unzip', '-q', '-o', zp, '-d', '/content/words_pilot_raw'], check=True)
src = os.path.dirname(glob.glob('/content/words_pilot_raw/**/manifest.csv', recursive=True)[0])
for f in glob.glob(src + '/*'):
    shutil.copy(f, '/content/words_pilot')
print(len(glob.glob('/content/words_pilot/*.png')), 'images,', len(glob.glob('/content/words_pilot/*.txt')), 'captions')

In [ ]:
# 3) Install the trainer (ai-toolkit). Safe to run again.
%cd /content
import os
if not os.path.isdir('/content/ai-toolkit'):
    !git clone https://github.com/ostris/ai-toolkit /content/ai-toolkit
%cd /content/ai-toolkit
# Colab runs Python 3.13 and ai-toolkit pins an old scipy that has no ready-made build: drop that pin.
!sed -i -E 's/^scipy[^#]*/scipy/' requirements.txt
!pip install -q -r requirements.txt
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 4) Write the training settings (klein 4B, 300 steps, checkpoints saved to Drive)
import yaml
ARCH = 'flux2_klein_4b'
cfg = yaml.safe_load(open('/content/ai-toolkit/config/examples/train_lora_flux_24gb.yaml'))
cfg['config']['name'] = 'hwdv1_words_pilot'
proc = cfg['config']['process'][0]
proc['training_folder'] = '/content/drive/MyDrive/flux_out'
proc['trigger_word'] = 'HWDV1'
proc['network']['linear'] = 32
proc['network']['linear_alpha'] = 32
proc['datasets'][0]['folder_path'] = '/content/words_pilot'
proc['datasets'][0]['resolution'] = [512]
proc['train']['steps'] = 300
proc['train']['lr'] = 1e-4
proc['save']['save_every'] = 150
proc['model'] = {'arch': ARCH,
                 'name_or_path': 'black-forest-labs/FLUX.2-klein-base-4B',
                 'quantize': True, 'quantize_te': True, 'low_vram': True}
proc['sample']['width'] = 512
proc['sample']['height'] = 512
proc['sample']['sample_every'] = 150
proc['sample']['prompts'] = ['HWDV1 handwritten Hindi word on white paper: खेलने']
yaml.safe_dump(cfg, open('/content/ai-toolkit/config/hwdv1_words_pilot.yaml', 'w'), allow_unicode=True)
print(yaml.safe_dump(cfg, allow_unicode=True)[:2500])

In [ ]:
# 5) Train. About 20 seconds per step on a T4, so 300 steps is roughly two hours.
# The progress line shows e.g. 19/300. Test images and checkpoints appear in Drive: flux_out/hwdv1_words_pilot
%cd /content/ai-toolkit
!python run.py config/hwdv1_words_pilot.yaml

When training ends, look at the images in `My Drive/flux_out/hwdv1_words_pilot/samples` and send me a few. The first one (step 0) is the untrained model, later ones show how it changes.

In [ ]:
# 6) Generate test words with the trained LoRA (distilled 4B model, 4 steps; low-memory mode for a T4)
import torch, glob, os
try:
    from diffusers import Flux2KleinPipeline
except ImportError:
    !pip install -q -U git+https://github.com/huggingface/diffusers
    from diffusers import Flux2KleinPipeline
out_dir = '/content/drive/MyDrive/flux_out/hwdv1_words_pilot'
final = os.path.join(out_dir, 'hwdv1_words_pilot.safetensors')
files = sorted(glob.glob(out_dir + '/*.safetensors'))
lora = final if os.path.exists(final) else files[-1]
print('using', lora)
pipe = Flux2KleinPipeline.from_pretrained('black-forest-labs/FLUX.2-klein-4B', torch_dtype=torch.bfloat16)
pipe.load_lora_weights(lora)
pipe.enable_model_cpu_offload()
words = ['घर','पानी','किताब','स्कूल','हमारा','बहुत','कृपया','विक्रेता','क्रेता','रजिस्ट्री','दस्तावेज़','संपत्ति']
os.makedirs('/content/gen', exist_ok=True)
for i, w in enumerate(words):
    img = pipe(f'HWDV1 handwritten Hindi word on white paper: {w}', num_inference_steps=4,
               generator=torch.Generator('cpu').manual_seed(i)).images[0]
    img.save(f'/content/gen/{i:02d}.png')
print('done')

In [ ]:
# 7) Read each word back with OCR and compare with what we asked for
!apt-get -qq install -y tesseract-ocr tesseract-ocr-hin > /dev/null
!pip -q install pytesseract python-Levenshtein
import pytesseract, Levenshtein
from PIL import Image
tot = ok = 0
for i, w in enumerate(words):
    got = pytesseract.image_to_string(Image.open(f'/content/gen/{i:02d}.png'), lang='hin', config='--psm 8').strip()
    cer = Levenshtein.distance(w, got) / max(len(w), 1)
    ok += cer <= 0.2; tot += 1
    print(f'{w:12s} -> {got:12s} error rate {cer:.2f}')
print(f'{ok}/{tot} words readable within 20% error')

In [ ]:
# 8) Contact sheet (saved to Drive too)
from PIL import Image
ims = [Image.open(f'/content/gen/{i:02d}.png').resize((256,256)) for i in range(len(words))]
sheet = Image.new('RGB', (256*4, 256*3), 'white')
for i, im in enumerate(ims):
    sheet.paste(im, ((i%4)*256, (i//4)*256))
sheet.save('/content/drive/MyDrive/hwdv1_pilot_sheet.png')
sheet